# Классификация обращений

Научим модель выбирать отдел по сообщению студента. Сравним два способа представить текст — слова и символьные фрагменты — и посмотрим, какие обращения вызывают ошибки.

Начнём с простой точки отсчёта: насколько полезно всегда отвечать одной категорией? Затем обучим классификатор и сравним его результат с этим правилом.

In [3]:
from pathlib import Path
import os
import json
import re
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Находим корень раздачи независимо от места запуска Jupyter.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data" / "requests.csv").exists())
DATA = ROOT / "data"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
np.random.seed(42)
plt.rcParams.update({"font.family": "DejaVu Sans", "figure.figsize": (8, 4)})


## Данные и сравнение

Используем корпус обращений из первой работы. Готовые части train, validation и test находятся в столбце `split`; `group_id` связывает перефразировки одной ситуации.

Модели по словам и символам будут обучаться на одном train. Для выбора используем macro-F1 на validation; при равенстве оставляем вариант words. К test перейдём после выбора. На этом наборе уже не будем подбирать настройки.

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

df = pd.read_csv(DATA / "requests.csv", keep_default_na=False)
train = df[df["split"].eq("train")].copy()
valid = df[df["split"].eq("validation")].copy()
test = df[df["split"].eq("test")].copy()
labels = sorted(df["label"].unique())
print({k: len(v) for k, v in {"train": train, "validation": valid, "test": test}.items()})

{'train': 360, 'validation': 120, 'test': 120}


### Как читать предсказания

В маленьком примере ниже известны настоящие категории и три ответа воображаемого классификатора. Найдите ошибку до запуска. Какую долю сообщений он направил правильно?

In [5]:
demo_true = np.array(["it", "library", "library"])
demo_pred = np.array(["it", "it", "library"])
display(pd.DataFrame({"true":demo_true, "predicted":demo_pred,
                      "correct":demo_true == demo_pred}))
print("Доля верных ответов:", np.mean(demo_true == demo_pred))

,true,predicted,correct
0,it,it,True
1,library,it,False
2,library,library,True


Доля верных ответов: 0.6666666666666666


Правильны два ответа из трёх. Ошибочное библиотечное обращение попало в ИТ. Дальше будем получать такие предсказания от обученной модели и смотреть не только на общий процент, но и на ошибки по категориям.

## 1. Разделяем ситуации, а не перефразировки

Проверьте, что одна группа не встречается сразу в двух частях корпуса. Сохраните множества `group_id` для train, validation и test в словарь `groups`, затем найдите пересечения каждой пары.

Посмотрите на число примеров каждой категории в каждой части. Все ли шесть отделов представлены? Почему пустого пересечения идентификаторов строк было бы недостаточно для проверки этого корпуса?

In [6]:
groups = {}
# Ваш код
groups["train"] = set(train["group_id"])
groups["validation"] = set(valid["group_id"])
groups["test"] = set(test["group_id"])

print("train & validation:", groups["train"] & groups["validation"])
print("train & test:", groups["train"] & groups["test"])
print("validation & test:", groups["validation"] & groups["test"])

display(train["label"].value_counts())
display(test["label"].value_counts())
display(valid["label"].value_counts())

train & validation: set()
train & test: set()
validation & test: set()


label
it             60
library        60
dormitory      60
study          60
admission      60
scholarship    60
Name: count, dtype: int64

label
it             20
library        20
dormitory      20
study          20
admission      20
scholarship    20
Name: count, dtype: int64

label
it             20
library        20
dormitory      20
study          20
admission      20
scholarship    20
Name: count, dtype: int64

In [7]:
assert set(groups) == {"train", "validation", "test"}
assert len(set.union(*groups.values())) == df["group_id"].nunique()

## 2. От трёх документов к TF-IDF

Для начала возьмём три коротких текста из лекции. Обучите на них `toy_vectorizer`, сохраните матрицу в `toy_matrix` и выведите словарь, IDF и матрицу. Маленький результат удобно показать через `toarray()`, чтобы видеть каждую координату.

Проверьте столбец «книга»: слово дважды встретилось в первой строке, но df по всему набору равно 2. Объясните различие этих счётчиков. Сопоставьте первый нормированный вектор с ручным расчётом.

Большая матрица основного корпуса останется разреженной; её целиком переводить в `toarray()` не нужно.

In [8]:
toy = ["книга книга библиотека", "книга продление", "общежитие оплата"]
# Ваш код
toy_vectorizer = TfidfVectorizer()

toy_matrix = toy_vectorizer.fit_transform(toy)

words = toy_vectorizer.get_feature_names_out()
print(words)

print("IDF:")
print(toy_vectorizer.idf_)

print("TF-IDF:")
print(toy_matrix.toarray())


['библиотека' 'книга' 'общежитие' 'оплата' 'продление']
IDF:
[1.69314718 1.28768207 1.69314718 1.69314718 1.69314718]
TF-IDF:
[[0.54935123 0.83559154 0.         0.         0.        ]
 [0.         0.60534851 0.         0.         0.79596054]
 [0.         0.         0.70710678 0.70710678 0.        ]]


In [9]:
assert toy_matrix.shape == (3, 5)
assert np.allclose(np.asarray(toy_matrix.multiply(toy_matrix).sum(axis=1)).ravel(), 1)
book_index = toy_vectorizer.vocabulary_["книга"]
assert np.isclose(toy_vectorizer.idf_[book_index], np.log(4 / 3) + 1)

## 3. Постоянный ответ как baseline

Обучите на train `DummyClassifier(strategy="most_frequent")`. Он будет возвращать одну категорию независимо от сообщения. Проверьте его на validation.

Напишите `score_predictions`, возвращающую словарь с ключами `accuracy` и `macro_f1`. Для F1 передавайте весь список `labels` и `zero_division=0`, чтобы одинаково учитывать все категории даже при отсутствии предсказаний некоторых из них. Результат baseline сохраните в `baseline_scores`.

Сколько правильных ответов можно было ожидать по распределению классов? Почему macro-F1 получилась заметно ниже accuracy?

In [10]:
def score_predictions(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    macro = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    
    result = {}
    result["accuracy"] = acc
    result["macro_f1"] = macro
    return result

# Ваш код
baseline = DummyClassifier(strategy="most_frequent")

baseline.fit(train["text"], train["label"])

baseline_pred = baseline.predict(valid["text"])

print(baseline_pred[0])


baseline_scores = score_predictions(valid["label"], baseline_pred)
print(baseline_scores)

admission
{'accuracy': 0.16666666666666666, 'macro_f1': 0.047619047619047616}


In [11]:
assert 0 <= baseline_scores["accuracy"] <= 1
assert 0 <= baseline_scores["macro_f1"] <= 1

## 4. Слова и символы

Постройте два Pipeline с одной и той же логистической регрессией. В words используйте `TfidfVectorizer(ngram_range=(1,2), max_features=5000)`. В chars — `TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), max_features=5000)`. Классификатор в обоих случаях: `LogisticRegression(C=1, max_iter=1000, random_state=42)`.

Обучите оба варианта на train и сравните их на validation. Сохраните конвейеры в `models`, оценки в `validation_scores`, имя выбранного варианта в `selected_name`, сам конвейер в `selected`. Основной критерий — macro-F1; при равенстве выбираем words.

Какой тип различий между обращениями может лучше сохранять символьное представление? Приведите пару словоформ или опечатку в качестве примера.

In [14]:
models = {}
validation_scores = {}
# Ваш код
words_vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
words_classifier = LogisticRegression(C=1, max_iter=1000, random_state=42)

words_model = Pipeline([
    ("tfidf", words_vectorizer),
    ("clf", words_classifier),
])

words_model.fit(train["text"], train["label"])

words_pred = words_model.predict(valid["text"])
words_scores = score_predictions(valid["label"], words_pred)

models["words"] = words_model
validation_scores["words"] = words_scores
print("words:", words_scores)

chars_vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=5000)
chars_classifier = LogisticRegression(C=1, max_iter=1000, random_state=42)

chars_model = Pipeline([
    ("tfidf", chars_vectorizer),
    ("clf", chars_classifier),
])

chars_model.fit(train["text"], train["label"])

chars_pred = chars_model.predict(valid["text"])
chars_scores = score_predictions(valid["label"], chars_pred)

models["chars"] = chars_model
validation_scores["chars"] = chars_scores
print("chars:", chars_scores)


words_f1 = validation_scores["words"]["macro_f1"]
chars_f1 = validation_scores["chars"]["macro_f1"]

if chars_f1 > words_f1:
    selected_name = "chars"
else:
    selected_name = "words"

selected = models[selected_name]


words: {'accuracy': 0.45, 'macro_f1': 0.4120481798829734}
chars: {'accuracy': 0.7416666666666667, 'macro_f1': 0.7429906658307709}


In [15]:
assert set(models) == {"words", "chars"}
assert selected_name in models
assert selected is models[selected_name]
assert all(0 <= s["macro_f1"] <= 1 for s in validation_scores.values())

## 5. Где модель ошиблась

Примените выбранный конвейер к test, сохраните предсказания в `pred_test`, а метрики — в `test_scores`. Постройте матрицу ошибок: настоящие классы по строкам, предсказанные по столбцам.

Создайте таблицу `errors` с ошибочными обращениями. Выберите до пяти разных `group_id` для подробного разбора. Если ошибок меньше, разберите доступные и добавьте отдельные контрольные сообщения, не меняя модель.

Сравните несколько вариантов одной ошибочной группы. Ошибка сохраняется при перефразировании или появляется только в части строк? Проверьте также три контрольных входа: пустую строку, «жжжжжжжжж» и «Не открывается виртуальная комната конференции». Что определяет ответ, когда текст не даёт ни одного известного признака?

In [28]:
# Ваш код
from sklearn.metrics import confusion_matrix

pred_test = selected.predict(test["text"])

test_scores = score_predictions(test["label"], pred_test)
print(test_scores)

matrix = confusion_matrix(test["label"], pred_test, labels=labels)

matrix_table = pd.DataFrame(matrix, index=labels, columns=labels)

display(matrix_table)

test_with_pred = test.copy()
test_with_pred["predicted"] = pred_test

is_wrong = test_with_pred["label"] != test_with_pred["predicted"]
errors = test_with_pred[is_wrong]

print( len(errors))
display(errors[["group_id", "text", "label", "predicted"]])

error_groups = errors["group_id"].unique()

chosen_groups = error_groups[:5]
print("Группы для разбора:", chosen_groups)

for group in chosen_groups:
    print()
    print("Группа:", group)
    rows = test_with_pred[test_with_pred["group_id"] == group]
    display(rows[["text", "label", "predicted"]])

control_texts = [
    "",
    "жжжжжжжжж",
    "Не открывается виртуальная комната конференции",
]

control_pred = selected.predict(control_texts)
control_proba = selected.predict_proba(control_texts)

for i in range(len(control_texts)):
    print()
    print("Текст:", repr(control_texts[i]))
    print("Ответ:", control_pred[i])
    print("Вероятности:", control_proba[i].round(3))

print()
print("Порядок отделов:", selected.classes_)

{'accuracy': 0.7916666666666666, 'macro_f1': 0.7651970897407003}


,admission,dormitory,it,library,scholarship,study
admission,20,0,0,0,0,0
dormitory,0,20,0,0,0,0
it,0,0,20,0,0,0
library,0,0,6,10,0,4
scholarship,0,0,0,0,20,0
study,5,0,6,4,0,5


25


,group_id,text,label,predicted
190,library-19,Не могу найти монографию в списке доступной ли...,library,it
191,library-19,Здравствуйте! Не могу найти монографию в списк...,library,it
192,library-19,"Подскажите, пожалуйста. Не могу найти монограф...",library,it
193,library-19,Не могу найти монографию в списке доступной ли...,library,it
194,library-19,Нужна консультация. Не могу найти монографию в...,library,it
195,library-20,Мне нужна помощь в подборе источников для курс...,library,it
196,library-20,Здравствуйте! Мне нужна помощь в подборе источ...,library,study
197,library-20,"Подскажите, пожалуйста. Мне нужна помощь в под...",library,study
198,library-20,Мне нужна помощь в подборе источников для курс...,library,study
199,library-20,Нужна консультация. Мне нужна помощь в подборе...,library,study


Группы для разбора: <StringArray>
['library-19', 'library-20', 'study-18', 'study-19', 'study-20']
Length: 5, dtype: str

Группа: library-19


...


Группа: study-19


,text,label,predicted
390,Как исправить ошибку в зачётной ведомости?,study,it
391,Здравствуйте! Как исправить ошибку в зачётной ...,study,it
392,"Подскажите, пожалуйста. Как исправить ошибку в...",study,it
393,Как исправить ошибку в зачётной ведомости? Буд...,study,it
394,Нужна консультация. Как исправить ошибку в зач...,study,it



Группа: study-20


,text,label,predicted
395,Где опубликован график экзаменационной сессии?,study,admission
396,Здравствуйте! Где опубликован график экзаменац...,study,admission
397,"Подскажите, пожалуйста. Где опубликован график...",study,admission
398,Где опубликован график экзаменационной сессии?...,study,admission
399,Нужна консультация. Где опубликован график экз...,study,admission



Текст: ''
Ответ: it
Вероятности: [0.163 0.153 0.201 0.186 0.107 0.189]

Текст: 'жжжжжжжжж'
Ответ: it
Вероятности: [0.163 0.153 0.201 0.186 0.107 0.189]

Текст: 'Не открывается виртуальная комната конференции'
Ответ: it
Вероятности: [0.145 0.205 0.249 0.112 0.16  0.129]

Порядок отделов: ['admission' 'dormitory' 'it' 'library' 'scholarship' 'study']


In [ ]:
assert len(pred_test) == len(test)
assert set(pred_test) <= set(labels)
assert len(errors) == int(np.sum(test["label"].to_numpy() != pred_test))

## Разбор результата

Укажите выбранное представление и сопоставьте validation с test. Прочитайте ошибки из нескольких разных групп: какие формулировки оказались трудными?

Выберите одно ошибочное обращение и предложите объяснение. Что нужно дополнительно посмотреть или изменить, чтобы проверить эту гипотезу? Отдельно объясните, что произошло бы при случайном разделении стилистических вариантов одной группы между train и test.

Сдайте выполненный ноутбук с таблицами, матрицей ошибок и своими пояснениями. Отдельный отчёт по тем же результатам не нужен.